# Step 4: Automated Production Trading Engine

## 1. Context & Objective
In the previous steps, we built the live data collection pipeline (`01_data_collection.ipynb`), engineered the lean indicator suite (`02_feature_engineering.ipynb`), and validated our strategy models with Level-2 execution simulation (`03_backtesting.ipynb`).

This notebook documents and validates **Step 4: The Production Trading Engine** (`src/engine/`).
The engine modularizes the verified research logic into a production-grade, event-driven, asynchronous execution system for Polymarket 5-minute binary prediction markets.

### Core Architecture
The engine is structured around a dual-class lifecycle model:
- **`TradingEngine`** (`src/engine/runner.py`): The top-level orchestrator managing rolling slot discovery, multi-market lifecycle management, session PnL accounting, risk circuit-breakers (`MAX_SESSION_LOSS`), and state persistence.
- **`MarketLifecycle`** (`src/engine/market_lifecycle.py`): The single-market state machine (`INIT` → `RUNNING` → `STOPPING` → `DONE`), driving order execution, fills, expirations, and Chainlink oracle settlement.
- **`StrategyContext`** (`src/engine/strategies/base.py`): The event-driven API connecting strategies to live order books, price tickers, and asynchronous order placement callbacks.

## 2. Setup & Dependencies
Import the modular engine classes, indicators, wallet tracker, and strategy modules.

In [1]:
import asyncio
import json
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

import sys
sys.path.insert(0, str(Path('../').resolve()))

from src.engine.types import (
    Order, OrderAction, OrderType, LifecycleState,
    OrderRequest, PendingOrder, CompletedOrder, BookSnapshot
)
from src.engine.wallet import WalletTracker
from src.engine.indicators import Indicators, RSI, ATR, RTV
from src.engine.client import SimTradingClient, is_sim_filled
from src.engine.user_channel import UserChannel
from src.engine.strategies import STRATEGIES, DEFAULT_STRATEGY
from src.engine.market_lifecycle import MarketLifecycle
from src.engine.runner import TradingEngine
from src.utils.slot import get_slot_timestamps, get_market_slug

print('All engine modules successfully imported!')
print(f'Registered Strategies: {list(STRATEGIES.keys())}')

All engine modules successfully imported!
Registered Strategies: ['late-entry', 'simulation']


## 3. Wallet & Balance Tracking (`WalletTracker`)
In live execution, capital management is critical. The `WalletTracker` manages:
1. **Cash Reservations**: Holds unreserved capital for pending buy orders (`lock_for_buy`), releasing upon cancellation or fill.
2. **Share Inventory**: Prevents short-selling by tracking unreserved share holdings for pending sell orders (`lock_for_sell`).
3. **Settlement Credits**: Accurately deposits \$1.00 per winning share and \$0.00 per losing share upon terminal market resolution.

In [2]:
# Demonstrate WalletTracker state management
wallet = WalletTracker(initial_balance=50.0, log_fn=print)

# 1. Place a limit buy order reservation
token_up = 'token_up_123'
wallet.lock_for_buy(order_id='order_1', price=0.49, shares=10.0, label='Test Buy')
print(f'Total Balance: ${wallet.balance:.2f} | Available Cash: ${wallet.available_cash:.2f}')

# 2. Buy fills: deduct cash, credit shares
wallet.on_buy_filled(order_id='order_1', token_id=token_up, price=0.49, filled_shares=10.0)
print(f'After Fill Cash: ${wallet.balance:.2f} | Holding: {wallet.total_shares(token_up)} shares')

# 3. Reserve shares for a take-profit sell
wallet.lock_for_sell(order_id='order_2', token_id=token_up, shares=10.0, label='Take Profit')
print(f'Available Unreserved Shares: {wallet.available_shares(token_up)}')

# 4. Sell fills: deduct shares, credit proceeds
wallet.on_sell_filled(order_id='order_2', token_id=token_up, price=0.70, filled_shares=10.0, fee=0.0)
print(f'After Sell Cash: ${wallet.balance:.2f} | Holding: {wallet.total_shares(token_up)} shares')

[Wallet] Initialized with balance: $50.00
[Wallet] Lock Buy (Test Buy): -$4.90 | Avail Cash: $45.10
Total Balance: $50.00 | Available Cash: $45.10
[Wallet] BUY Filled: -$4.90 cash, +10.00 shares | Cash: $45.10 | Holding: 10.00 shares
After Fill Cash: $45.10 | Holding: 10.0 shares
[Wallet] Lock Sell (Take Profit): -10.00 shares | Avail Shares: 0.00
Available Unreserved Shares: 0.0
[Wallet] SELL Filled: +$7.00 cash (fee $0.0000), -10.00 shares | Cash: $52.10 | Holding: 0.00 shares
After Sell Cash: $52.10 | Holding: 0.0 shares


## 4. Streaming Indicator Engine (`Indicators`)
The engine updates real-time streaming indicators on each market tick without batch recomputation:
- **Price Gap**: Real-time Binance spot BTC minus Chainlink Oracle "price to beat".
- **Wilder RSI(14)**: Measures gap expansion / momentum and suppresses premature stop-losses.
- **Wilder ATR(14)**: Volatility filter (`ATR <= 2.0` required for late entry).
- **Gap Safety**: Ratio $|\text{gap}| / \text{ATR}$ (`>= 40.0` required for late entry).
- **Peak Gap Ratio**: Ratio $|\text{gap}| / \text{peakAbsGap}$ (`>= 0.75` required for late entry).

In [3]:
indicators = Indicators()

# Simulate streaming price updates
sim_prices = [75000 + (i * 3.5) + ((-1)**i * 1.5) for i in range(25)]
price_to_beat = 74950.0

print('Streaming 25 ticks into indicator engine:')
for i, p in enumerate(sim_prices):
    gap = p - price_to_beat
    indicators._last_update_ms = 0  # bypass throttle for demo
    indicators.tick(gap=gap, btc_price=p)

print(f'  Current Price:    ${sim_prices[-1]:,.2f}')
print(f'  Price to Beat:    ${price_to_beat:,.2f}')
print(f'  Price Gap:        ${gap:+.2f}')
print(f'  Wilder RSI(14):   {indicators.rsi:.2f}')
print(f'  Wilder ATR(14):   ${indicators.atr:.2f}')
print(f'  Gap Safety:       {indicators.gap_safety(gap):.2f}')
print(f'  Peak Gap Ratio:   {indicators.peak_gap_ratio(gap):.2f}')

Streaming 25 ticks into indicator engine:
  Current Price:    $75,085.50
  Price to Beat:    $74,950.00
  Price Gap:        $+135.50
  Wilder RSI(14):   100.00
  Wilder ATR(14):   $3.56
  Gap Safety:       38.08
  Peak Gap Ratio:   1.00


## 5. Strategy Models
The engine includes two event-driven strategy implementations:

### 1. Late-Entry Strategy (`late-entry`)
- **Entry Window**: Late in market window ($T \le 90$s remaining).
- **Regime Filters**: $\text{ATR} \le 2.0$, $\text{Gap Safety} \ge 40.0$, $\text{Peak Gap Ratio} \ge 0.75$, $\text{Ask} > 0.85$, $\text{Liquidity} \ge \$20$.
- **Protection**: Dynamic stop-loss at $0.48$ with RSI and directional gap momentum suppression.

### 2. Limit Order Simulation Strategy (`simulation`)
- **Entry**: Discount maker limit buy at $0.49$ on market open.
- **Target**: Take-profit limit sell at $0.70$ upon buy fill.
- **Emergency Exit**: Emergency market sell at best bid if unfilled with $30$s remaining.

In [4]:
# Inspect strategy functions
for name, fn in STRATEGIES.items():
    print(f'Strategy: {name:15} | Handler: {fn.__name__} | Module: {fn.__module__}')

Strategy: late-entry      | Handler: late_entry_strategy | Module: src.engine.strategies.late_entry
Strategy: simulation      | Handler: simulation_strategy | Module: src.engine.strategies.simulation


## 6. Live Paper-Trading Session Demonstration
Here we demonstrate how the `TradingEngine` initializes and executes rolling market rounds in simulation mode.
The engine subscribes to live Binance ticker feeds, initializes the simulated CLOB client, and progresses active markets through `INIT` → `RUNNING` → `STOPPING` → `DONE`.

In [5]:
# Demonstrate TradingEngine parameterization and initialization
engine = TradingEngine(
    strategy_name='simulation',
    slot_offset=1,
    prod=False,
    rounds=1,
    asset='btc',
    window='5m',
    wallet_balance=50.0,
    max_session_loss=3.0
)

print('Trading Engine instantiated with configuration:')
print(f'  Strategy:          {engine.strategy_name}')
print(f'  Mode:              {"PROD" if engine.prod else "SIMULATION"}')
print(f'  Asset / Window:    {engine.asset.upper()} / {engine.window}')
print(f'  Slot Offset:       +{engine.slot_offset}')
print(f'  Initial Balance:   ${engine.wallet.balance:.2f}')
print(f'  Max Session Loss:  -${engine.max_session_loss:.2f}')
print(f'  State Path:        {engine.state_file}')

Trading Engine instantiated with configuration:
  Strategy:          simulation
  Mode:              SIMULATION
  Asset / Window:    BTC / 5m
  Slot Offset:       +1
  Initial Balance:   $50.00
  Max Session Loss:  -$3.00
  State Path:        D:\Companies\chaolin\Research\polymarket_trading_bot\state\engine_sim.json


## 7. Command-Line Interface & Deployment
The production engine is operated via the top-level launcher `run_engine.py`:

```powershell
# 1. Run Late-Entry Strategy in paper-trading simulation (5 rounds):
python run_engine.py --strategy late-entry --rounds 5 --slot-offset 1

# 2. Run continuous simulation indefinitely until stopped (Ctrl+C safely flushes state):
python run_engine.py --strategy late-entry --rounds 0

# 3. Run Simulation strategy with custom cash balance & loss limit:
python run_engine.py --strategy simulation --wallet-balance 100 --max-session-loss 5

# 4. Run in production mode with real funds on Polygon (requires confirmation prompt):
python run_engine.py --strategy late-entry --prod
```

### Circuit Breakers & Risk Safeguards
- **`MAX_SESSION_LOSS`**: If cumulative losses reach the threshold (e.g. $-\$3.00$), the engine halts further entries and initiates an orderly shutdown.
- **State Persistence**: Current session metrics and active positions are saved to `state/engine_sim.json` every 5 seconds for crash resilience.
- **Graceful Signal Handling**: `SIGINT` (Ctrl+C) cancels in-flight buys, allows active positions to settle, and flushes session accounting to disk.

## 8. Summary of Completed 4-Step Pipeline

| Step | Deliverable | Description | Status |
| :--- | :--- | :--- | :---: |
| **Step 1** | `notebooks/01_data_collection.ipynb` & `run_collector.py` | Full L2 depth capture, Gamma metadata, Chainlink baseline/settlement, Binance ticker. | **Completed** |
| **Step 2** | `notebooks/02_feature_engineering.ipynb` | Lean indicator suite: Gap, RSI-14, ATR-14, RTV-30, Gap Safety, Peak Gap Ratio, OBI. | **Completed** |
| **Step 3** | `notebooks/03_backtesting.ipynb` | L2 book walking fill simulator, Late-Entry & Simulation strategies, Chainlink settlement accounting. | **Completed** |
| **Step 4** | `notebooks/04_production_engine.ipynb` & `src/engine/` | Dual-class execution engine (`TradingEngine` & `MarketLifecycle`), event-driven order routing, circuit-breaker risk management. | **Completed** |

The complete research-to-production trading system is now fully implemented, modularized, and ready for deployment.